# Пилот GRPO: время шага и весь путь на настоящих данных

Значения «по пилоту» из `docs/grpo.md` фиксируются до основных запусков. Проверки из `grpo.ipynb` (раздел 3) задают
предел контекста, нагрузку карты 1 и порог расхождения log p; здесь — то, что видно только на настоящих шагах:
время сбора, награды, обучения, публикации адаптера и validation, и по нему — хватает ли времени на 150 шагов.
Заодно проходит весь путь: журналы, адаптер шага в vLLM, чекпоинт, сохранение состояния.

3 шага, validation на третьем; папка — `runs/grpo_pilot/` (кривые и выбор чекпоинта её не видят). Нужно: vLLM-сервер
с исходной T-lite на карте 1 и пройденные проверки — разделы 1–2 `grpo.ipynb`. Карта 0 должна быть свободна.

In [1]:
import json, os, statistics, sys, time
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # до первого обращения к CUDA (docs/grpo.md, раздел 8)
from datetime import datetime
from pathlib import Path

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))  # если пакет не установлен через pip install -e .
from presentation_rl.training.common import RLConfig, read_journal
from presentation_rl.training.vllm_client import VLLMClient

os.environ["WIKI_PROXY"] = ""  # Википедия открыта в пространстве; туннель — docs/server.md
config = RLConfig.load(ROOT / "configs" / "algorithms" / "grpo.yaml")  # основной конфиг — для оценки времени
pilot = RLConfig.load(ROOT / "configs" / "algorithms" / "grpo.yaml", steps=3, eval_every=3, runs_dir="runs/grpo_pilot")
client = VLLMClient(pilot.vllm_url)
print("vLLM", client.version(), client.models())
print("порог расхождения log p:", pilot.max_logprob_mismatch, "(None — пилот не останавливается, только пишет в журнал)")

mkdir -p failed for path /home/jovyan/.config/matplotlib: [Errno 20] Not a directory: '/home/jovyan/.config/matplotlib'
Matplotlib created a temporary cache directory at /tmp/matplotlib-zg7yw69z because there was an issue with the default path (/home/jovyan/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


vLLM 0.8.5 ['base']
порог расхождения log p: {'mean_abs': None, 'p99': None} (None — пилот не останавливается, только пишет в журнал)


## Запуск

Около 3 шагов сбора по 64 эпизода и одна validation (32 эпизода). Если упадёт — ошибка и шаг в `events.jsonl`
папки пилота; продолжать не нужно, после исправления — новый пилот.

In [2]:
from presentation_rl.training.grpo import train

STARTED = time.time()
RUN = train(pilot, ROOT, client=client)
print(RUN, f"всего {(time.time() - STARTED) / 60:.1f} мин")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

/home/jovyan/Diplom/runs/grpo_pilot/20261113_0008_seed0 всего 76.4 мин


## Время и оценка полного запуска

По шагам: сбор эпизодов (в нём — ожидание Википедии), награда, обучение, публикация адаптера. Время validation —
от записи шага 3 до отметки чекпоинта (`done.json`, с точностью до секунды). Первый шаг дольше: кеш статей ещё
не прогрет, поэтому оценка полного запуска — по шагам 2–3 и, для сравнения, по всем.

In [3]:
RUN = RUN if "RUN" in globals() else sorted((ROOT / pilot.runs_dir).glob("*_seed*"))[-1]
steps = [r for r in read_journal(RUN / "steps.jsonl") if "step" in r]


def stamp(text):
    return datetime.fromisoformat(text).timestamp()


def duration(r):  # сбор (с ожиданием Википедии внутри) + награда + обучение + адаптер
    return sum(v for k, v in r["seconds"].items() if k != "network")


for r in steps:
    t = r["seconds"]
    print(f"шаг {r['step']}: сбор {t['generation']:.0f} с (Википедия {t['network']:.0f}), награда {t['reward']:.0f}, "
          f"обучение {t['train']:.0f}, адаптер {t['adapter']:.0f} | эпизодов {r['episodes']}, обучающих токенов "
          f"{r['tokens']}, пик памяти {r['peak_memory_gb']:.1f} ГБ | завершения {r['finish']}, "
          f"обрезанных выдач {r['truncated_outputs']}")
    print(f"        награда: поиск {r.get('search_reward')}, без поиска {r.get('plain_reward')}; доля поиска "
          f"{r.get('search_share')}; оборваны по контексту: поиск {r.get('search_finish_context')}, без "
          f"{r.get('plain_finish_context')}; KL k3 {r['kl_ref_k3']:.4g}; расхождение log p {r['logprob_mismatch']}")

done = json.loads((RUN / "checkpoints" / f"step_{steps[-1]['step']:04d}" / "done.json").read_text(encoding="utf-8"))
VAL = stamp(done["time"]) - stamp(steps[-1]["time"])
if "STARTED" in globals():
    print(f"подготовка (модели, снимок кеша, первый адаптер): "
          f"{(stamp(steps[0]['time']) - duration(steps[0]) - STARTED) / 60:.1f} мин")
validations = -(-config.steps // config.eval_every)
print(f"validation: {VAL / 60:.1f} мин; в полном запуске их {validations}")
estimates = [("по шагам 2–3", steps[1:]), ("по всем шагам", steps)] if len(steps) > 1 else [("по шагу 1", steps)]
for name, part in estimates:
    per_step = statistics.mean(duration(r) for r in part)
    hours = (per_step * config.steps + VAL * validations) / 3600
    print(f"{name}: шаг {per_step / 60:.1f} мин → {config.steps} шагов ≈ {hours:.1f} ч на запуск, "
          f"≈ {2 * hours:.1f} ч на два сида")

шаг 1: сбор 561 с (Википедия 80), награда 18, обучение 401, адаптер 30 | эпизодов 64, обучающих токенов 56375, пик памяти 22.6 ГБ | завершения {'stop': 64}, обрезанных выдач 0
        награда: поиск 0.7193410764030173, без поиска 0.8028071901492883; доля поиска 0.875; оборваны по контексту: поиск 0.0, без 0.0; KL k3 0; расхождение log p {'tokens': 56375, 'mean': -2.6943696336636755e-05, 'mean_abs': 0.003295913389472024, 'p99': 0.028946518898010254}
шаг 2: сбор 765 с (Википедия 110), награда 25, обучение 445, адаптер 31 | эпизодов 64, обучающих токенов 67067, пик памяти 22.4 ГБ | завершения {'stop': 64}, обрезанных выдач 0
        награда: поиск 0.5262006700576328, без поиска 0.749161136329536; доля поиска 1.0; оборваны по контексту: поиск 0.0, без 0.0; KL k3 2.153e-05; расхождение log p {'tokens': 67067, 'mean': -7.896318199226342e-05, 'mean_abs': 0.003125271235719958, 'p99': 0.029647111892700195}
шаг 3: сбор 618 с (Википедия 100), награда 18, обучение 411, адаптер 24 | эпизодов 64, об